In [1]:
pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 26.5 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import mysql.connector
from mysql.connector import errorcode

DB_CONFIG = {
    'user': 'root',
    'password': 'Sql4my%25', 
    'host': '127.0.0.1',
    'database': 'Big_Bear_Jet',
    'raise_on_warnings': True
}

def get_pilot_roster():
    """
    Connects to the DB and retrieves the upcoming pilot roster.
    """
    try:
        # 1. Establish the connection
        cnx = mysql.connector.connect(**DB_CONFIG)
        cursor = cnx.cursor()

        # 2. Define and execute the query
        query = """
            SELECT 
                f.ScheduledDeparture,
                f.OriginAirport,
                f.DestinationAirport,
                p.FirstName,
                p.LastName,
                fa.Role
            FROM 
                FlightAssignment fa
            JOIN 
                Pilot p ON fa.PilotID = p.PilotID
            JOIN 
                Flight f ON fa.FlightID = f.FlightID
            WHERE 
                f.ScheduledDeparture > NOW() -- Only show future flights
            ORDER BY 
                f.ScheduledDeparture, f.FlightID, fa.Role DESC;
        """
        cursor.execute(query)

        # 3. Fetch all results
        print("--- ✈️ Big Bear Jet: Upcoming Pilot Roster ---")
        for (dept, origin, dest, first, last, role) in cursor:
            print(
                f"[{dept.strftime('%Y-%m-%d %H:%M')}] {origin} -> {dest} | "
                f"{role}: {first} {last}"
            )

    except mysql.connector.Error as err:
        if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
            print("Something is wrong with your user name or password")
        elif err.errno == errorcode.ER_BAD_DB_ERROR:
            print("Database does not exist")
        else:
            print(err)
    finally:
        # 4. Close connections
        if 'cursor' in locals() and cursor:
            cursor.close()
        if 'cnx' in locals() and cnx:
            cnx.close()

if __name__ == '__main__':
    get_pilot_roster()

--- ✈️ Big Bear Jet: Upcoming Pilot Roster ---


In [4]:
import pandas as pd
import mysql.connector
from mysql.connector import errorcode

In [5]:
cnx = mysql.connector.connect(**DB_CONFIG)
    

query1 = """
        SELECT 
            c.Name,
            c.CompanyName,
            COUNT(f.FlightID) AS TotalFlights,
            SUM(f.Revenue) AS TotalRevenue
        FROM Customer c
        JOIN FlightRequest fr ON c.CustomerID = fr.CustomerID
        JOIN Flight f ON fr.RequestID = f.RequestID
        GROUP BY c.CustomerID, c.Name, c.CompanyName
        ORDER BY TotalRevenue DESC
        LIMIT 5;
    """
    
customer_df = pd.read_sql_query(query1, cnx)

if 'cnx' in locals() and cnx.is_connected():
        cnx.close()

customer_df

/var/folders/vl/z6gfc_1s4mbd9k5knjb6qm1c0000gn/T/ipykernel_33451/658071260.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customer_df = pd.read_sql_query(query1, cnx)


,Name,CompanyName,TotalFlights,TotalRevenue
0,Peter Rodriguez,None,1,45050.0
1,Mia Jackson,None,2,20160.0
2,Dr. Aris Thorne,Warren Wilson College,2,19080.0
3,The Kim Family,None,1,18000.0
4,David Green,Summit Ventures,1,18000.0


In [6]:
try:
    cnx = mysql.connector.connect(**DB_CONFIG)
    
    query2 = """
        SELECT 
            p.PilotID,
            p.FirstName,
            p.LastName,
            p.QualifiedAircraftType,
            COUNT(fa.FlightID) AS AssignedFlightCount
        FROM Pilot p
        LEFT JOIN FlightAssignment fa ON p.PilotID = fa.PilotID
        GROUP BY p.PilotID
        ORDER BY AssignedFlightCount ASC;
    """
    
    pilot_df = pd.read_sql_query(query2, cnx)

finally:
    if 'cnx' in locals() and cnx.is_connected():
        cnx.close()

pilot_df

/var/folders/vl/z6gfc_1s4mbd9k5knjb6qm1c0000gn/T/ipykernel_33451/136712991.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pilot_df = pd.read_sql_query(query2, cnx)


,PilotID,FirstName,LastName,QualifiedAircraftType,AssignedFlightCount
0,4,Sarah,Wilson,Cessna 172 Skyhawk,0
1,5,David,Brown,"Cessna 172 Skyhawk, Beechcraft Baron G58",0
2,6,Jessica,Miller,Cirrus SR22,0
3,7,James,Lee,Pilatus PC-12,0
4,11,William,Jackson,Embraer Phenom 300,0
5,12,Olivia,White,Cessna 172 Skyhawk,0
6,13,Richard,Harris,"Daher TBM 900, Pilatus PC-12",0
7,16,Ava,Martinez,Beechcraft Baron G58,0
8,17,Joseph,Robinson,Cirrus Vision SF50,0
9,10,Megan,Thomas,"Gulfstream G650, Gulfstream G700",1


In [7]:
# Cell 4: Run Query 3 (Maintenance Dashboard)
try:
    cnx = mysql.connector.connect(**DB_CONFIG)
    
    query3 = """
        SELECT a.TailNumber, a.AircraftType, me.MaintenanceType, t.FirstName
        FROM MaintenanceEvent me
        JOIN Aircraft a ON me.TailNumber = a.TailNumber
        JOIN Technician t ON me.TechnicianID = t.TechnicianID
        WHERE me.Status = 'In-Progress';
    """
    
    maint_df = pd.read_sql_query(query3, cnx)

finally:
    if 'cnx' in locals() and cnx.is_connected():
        cnx.close()

maint_df

/var/folders/vl/z6gfc_1s4mbd9k5knjb6qm1c0000gn/T/ipykernel_33451/1918668562.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  maint_df = pd.read_sql_query(query3, cnx)


,TailNumber,AircraftType,MaintenanceType,FirstName
0,N525CJ,Cessna Citation Jet,100-Hour Inspection,Mark
1,N423CS,Cirrus SR22,Propeller Balance,Chris
2,N50SF,Cirrus Vision SF50,Avionics Update,Emily
